# brain-model: TRIBE v2 pipeline on Colab

**Before running any cell:**
1. Runtime > Change runtime type > **A100 GPU** (T4/V100 only have 16 GB VRAM and will OOM on TRIBE)
2. **Skip Cell 3** (HuggingFace login) — running with `--no-text` since Llama 3.2-3B access is pending

**What this does (simple version):**
The GPU step is running Meta's pre-trained TRIBE v2 brain model on 50 short videos to get
predicted brain responses per video. This is *inference*, not training — TRIBE's weights are fixed.
After that, Ridge regression on the resulting features takes ~5 min on CPU.

**Note on --no-text:** TRIBE uses video + audio + text (Llama). Without text, predictions use
video + audio only. The A vs B comparison stays fair because both sides are missing text equally.
Re-run with text once Llama access is approved for the full preregistered result.

| Cell | What it does | Approx time |
|---|---|---|
| 0 | GPU check | instant |
| 1 | Mount Drive + clone repo | 1 min |
| 2 | Install dependencies | ~5 min |
| 3 | ~~HuggingFace login~~ **SKIP** (Llama pending) | — |
| 4 | Download 50 videos (~150 MB) | ~2 min |
| 5 | Smoke test on one video | ~2 min |
| 6a | Throughput check (5 clips) | ~10 min |
| 6b | TRIBE extraction (50 clips, --no-text) | 1–4 hrs |
| 6c | Monitor progress | instant |
| 7 | Features + CV experiments + report | ~5 min |
| **8** | **Train final B_pca8 model** | ~1 min |
| **9** | **Upload an ad + predict engagement** | ~2 min |
| 10 | Download outputs to Mac (optional) | varies |

In [ ]:
# Cell 0 — GPU check
# Run this first. TRIBE v2 OOMs on T4/V100 (16 GB). A100 (40 GB) is required.
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"GPU: {gpu}")
assert "A100" in gpu, "Wrong GPU. Go to Runtime > Change runtime type > A100 GPU."

In [ ]:
# Cell 1 — Mount Google Drive + clone repo
# Data goes into Drive so it survives session restarts.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO = "/content/drive/MyDrive/brain-model"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/anu-kap/brain-model.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

%cd /content/drive/MyDrive/brain-model
print("Working directory:", os.getcwd())

In [ ]:
# Cell 2 — Install dependencies (~5 min the first time; safe to re-run)
!apt-get install -y -qq ffmpeg
!pip install -q uv

# Pin torch + torchaudio to the same build; --force-reinstall replaces Colab's
# pre-installed torchaudio which may be built against a different torch ABI.
!pip install -q --force-reinstall \
    "torch==2.5.1+cu124" \
    "torchvision==0.20.1+cu124" \
    "torchaudio==2.5.1+cu124" \
    --index-url https://download.pytorch.org/whl/cu124

!pip install -q -e ".[gpu,dev]" --ignore-requires-python
!pip install -q "exca>=0.5.20,<0.5.26" --ignore-requires-python

# Verify — must show torchaudio 2.5.1
!python -c "import torch, torchaudio, tribev2, exca; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| cuda', torch.cuda.is_available(), '| exca', exca.__version__)"
!uvx whisperx --help > /dev/null 2>&1 && echo "whisperx ok" || echo "WARNING: uvx whisperx failed"

In [ ]:
# Cell 3 — HuggingFace login — SKIP FOR NOW
# Llama 3.2-3B access is pending. We're running with --no-text (video + audio only).
# Come back here once access is approved, then re-run cells 5 and 6b without --no-text.
#
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get("HF_TOKEN"))
print("Cell 3 skipped — running without text stream (--no-text).")

In [ ]:
# Cell 4 — Download the first 50 SnapUGC videos from CDN links (~150 MB)
# Idempotent: skips files that already exist in data/videos/.
!python scripts/01_build_subset.py --from-manifest results/manifest.csv

In [ ]:
# Cell 5 — Smoke test: verify TRIBE works end-to-end on one video
# Checks: preds.shape == (T, 20484), embedding cache readable, all values finite.
# The key unverified assumption is the embedding cache read-back (tribe_extract.py:83).
#
# --no-text: skip Llama/WhisperX (Llama access pending); uses video + audio only.
# If OOM: add -u data.video_feature.image.batch_size=2
import pandas as pd
m = pd.read_parquet('data/manifest.parquet')
video = m.path.iloc[0]
print(f"Testing on: {video}")
!python scripts/00_smoke_test.py --video {video} --no-text

In [ ]:
# Cell 6a — Throughput check: time TRIBE on 5 clips
# Look at 'total_s' per clip in the log output.
# If consistently > 180 s/clip, reduce --limit 50 to a smaller number in Cell 6b.
!python scripts/02_extract_tribe.py --limit 5 --no-text

In [ ]:
# Cell 6b — TRIBE extraction over 50 videos (~1–4 hrs)
#
# --no-text: video + audio only (Llama access pending)
# --limit 50: process first 50 from the manifest
#
# RESUMABLE: if the session times out, re-run this cell after reconnecting.
# The script skips videos that already have data/preds/<id>.npy + .meta.json.
# All data lives in Google Drive and is safe across restarts.
#
# Monitor progress with Cell 6c (open in a separate tab).
!python scripts/02_extract_tribe.py --no-text --limit 50 2>&1 | tee data/extract.out

In [ ]:
# Cell 6c — Check extraction progress (run any time while 6b is running)
import glob
done = len(glob.glob('data/preds/*.npy'))
print(f"{done}/50 videos extracted")
!tail -5 data/extract.out 2>/dev/null || echo "(no log yet)"

In [ ]:
# Cell 7 — Build features + run CV experiments + generate report (~5 min, CPU)
# Run after extraction finishes (or after copying data/preds + data/embeddings from another machine).
# The --quick flag does 2 repeats x 3 draws as a sanity check; the full run does 10 x 10.
!python scripts/03_build_features.py
!python scripts/04_run_experiments.py --quick  # sanity check first
!python scripts/04_run_experiments.py          # full preregistered protocol
!python scripts/05_report.py
print("\nDone. See results/summary.md for the verdict.")

In [ ]:
# Cell 8 — Download TRIBE outputs to your Mac (optional)
# Use this to run steps 03-05 locally instead of in Colab.
# Zips data/preds + data/embeddings (~1 GB total).
from google.colab import files
!zip -r /content/tribe_outputs.zip data/preds data/embeddings
files.download('/content/tribe_outputs.zip')

In [ ]:
# Cell 8 — Train the final model on all extracted videos
# Fits B_pca8 (simple + PCA-8 embeddings + Yeo-7 brain features) on all available data.
# Saves results/model.joblib for use in Cell 9.
# Run after Cell 7 (features + experiments must be built first).
!python scripts/06_train_final_model.py

In [ ]:
# Cell 9 — Upload a new ad video and predict engagement
# Upload any mp4/mov file from your computer, then run TRIBE + predict.
# Requires the GPU runtime and results/model.joblib from Cell 8.
from google.colab import files
import os

print("Upload your ad video (mp4, mov, etc.)...")
uploaded = files.upload()
ad_path = list(uploaded.keys())[0]
print(f"Uploaded: {ad_path}")

# Run TRIBE on it and predict ECR
!python scripts/07_predict.py --video "{ad_path}" --no-text